###  Imports and configuration

In [11]:
"""Token-based replay fitness of each policy's simulated logs."""

from pathlib import Path

import pandas as pd
import pm4py
from scipy import stats

# Configuration
CONFIG = "flex"       # flex, noflex_matrix, ...
NOISE = 0.2           # Inductive Miner noise threshold

LOGS = ["AcademicCredentials", "BPIC_2012", "BPIC_2017"]
POLICIES = ["RA-RR", "DM-RR", "DM-GR", "DM-DRL", "DRL-DRL"]

ROOT = Path("../../data/evaluation_results") / CONFIG
OUT_DIR = Path("results")

print(f"Configuration: {CONFIG}")
print(f"Noise threshold: {NOISE}")

Configuration: flex
Noise threshold: 0.2


###  Helper functions

In [12]:
def to_event_log(path: Path) -> pd.DataFrame:
    """Load a CSV and convert it into a PM4Py event log."""

    df = pd.read_csv(
        path,
        usecols=["case_id", "activity", "end_time"]
    )

    df["case_id"] = df["case_id"].astype(str)
    df["activity"] = df["activity"].astype(str).str.strip()
    df["end_time"] = pd.to_datetime(
        df["end_time"],
        format="mixed"
    )

    return pm4py.format_dataframe(
        df,
        case_id="case_id",
        activity_key="activity",
        timestamp_key="end_time"
    )


def ci95(values: pd.Series) -> float:
    """Calculate the 95% confidence interval half-width."""

    n = len(values)

    if n < 2:
        return 0.0

    return (
        stats.t.ppf(0.975, n - 1)
        * values.std(ddof=1)
        / n**0.5
    )

###  Discover Petri nets and evaluate training logs

In [13]:
rows = []

# Store discovered models for each event log
models = {}

for log in LOGS:

    train_path = Path(f"../../data/logs/{log}/{log}_train.csv")
    train = to_event_log(train_path)

    # Discover Petri net using Inductive Miner
    net, im, fm = pm4py.discover_petri_net_inductive(
        train,
        noise_threshold=NOISE
    )

    models[log] = (net, im, fm)

    # Evaluate training log
    ref = pm4py.fitness_token_based_replay(
        train,
        net,
        im,
        fm
    )

    rows.append({
        "log": log,
        "policy": "TRAIN",
        "run": "train",
        **ref
    })

    print(
        f"[{log}] Model discovered; "
        f"train fitness={ref['log_fitness']:.4f}"
    )

replaying log with TBR, completed traces :: 100%|██████████| 53/53 [00:00<00:00, 716.24it/s]


[AcademicCredentials] Model discovered; train fitness=0.9884


replaying log with TBR, completed traces :: 100%|██████████| 735/735 [00:01<00:00, 706.85it/s] 


[BPIC_2012] Model discovered; train fitness=0.9998


replaying log with TBR, completed traces :: 100%|██████████| 1840/1840 [00:02<00:00, 626.82it/s]

[BPIC_2017] Model discovered; train fitness=0.9954


### Evaluate simulated logs for each policy

In [14]:
for log in LOGS:

    net, im, fm = models[log]

    print(f"\nEvaluating {log}")

    for policy in POLICIES:

        runs = sorted(
            (ROOT / log / policy / "simulated_logs")
            .glob("sim_run_*.csv")
        )

        if not runs:
            print(f"  [skip] {policy}: no simulated logs")
            continue

        policy_fitness = []

        for run in runs:

            simulated_log = to_event_log(run)

            fit = pm4py.fitness_token_based_replay(
                simulated_log,
                net,
                im,
                fm
            )

            rows.append({
                "log": log,
                "policy": policy,
                "run": run.stem,
                **fit
            })

            policy_fitness.append(fit["log_fitness"])

        # Calculate mean and confidence interval
        fitness_series = pd.Series(policy_fitness)

        mean_fitness = fitness_series.mean()
        confidence_interval = ci95(fitness_series)

        print(
            f"  {policy:8s} "
            f"fitness={mean_fitness:.4f} "
            f"± {confidence_interval:.4f} "
            f"({len(runs)} runs)"
        )


Evaluating AcademicCredentials


replaying log with TBR, completed traces :: 100%|██████████| 63/63 [00:00<00:00, 634.44it/s]


  RA-RR    fitness=0.9818 ± 0.0014 (10 runs)


replaying log with TBR, completed traces :: 100%|██████████| 22/22 [00:00<00:00, 745.12it/s]


  DM-RR    fitness=0.9798 ± 0.0009 (10 runs)


replaying log with TBR, completed traces :: 100%|██████████| 21/21 [00:00<00:00, 648.82it/s]


... (21 outputs left out)

replaying log with TBR, completed traces :: 100%|██████████| 2234/2234 [00:03<00:00, 693.99it/s]


  DM-DRL   fitness=1.0000 ± 0.0000 (10 runs)


replaying log with TBR, completed traces :: 100%|██████████| 20/20 [00:00<00:00, 213.92it/s]

  DRL-DRL  fitness=1.0000 ± 0.0000 (10 runs)


###  Save per-run results

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

per_run = pd.DataFrame(rows)

per_run_path = OUT_DIR / f"{CONFIG}_fitness_runs.csv"

#per_run.to_csv(per_run_path, index=False)

print(f"Per-run results saved to: {per_run_path}")

display(per_run.head())

Per-run results saved to: results\flex_fitness_runs.csv


,log,policy,run,perc_fit_traces,average_trace_fitness,log_fitness,percentage_of_fitting_traces
0,AcademicCredentials,TRAIN,train,84.170854,0.988815,0.988447,84.170854
1,BPIC_2012,TRAIN,train,99.669967,0.999859,0.999812,99.669967
2,BPIC_2017,TRAIN,train,89.259660,0.993554,0.995365,89.259660
3,AcademicCredentials,RA-RR,sim_run_01,71.356784,0.986430,0.982606,71.356784
4,AcademicCredentials,RA-RR,sim_run_02,72.613065,0.986291,0.981986,72.613065


###  Aggregate results

In [ ]:
metrics = [
    "log_fitness",
    "average_trace_fitness",
    "percentage_of_fitting_traces"
]

agg = (
    per_run
    .groupby(["log", "policy"], sort=False)[metrics]
    .agg(["mean", ci95])
    .reset_index()
)

# Flatten multi-level column names
agg.columns = [
    "_".join(c).rstrip("_")
    for c in agg.columns
]

agg_path = OUT_DIR / f"{CONFIG}_fitness_aggregated.csv"

#agg.to_csv(agg_path, index=False)

print(f"Aggregated results saved to: {agg_path}")

display(agg)

Aggregated results saved to: results\flex_fitness_aggregated.csv


,log,policy,log_fitness_mean,log_fitness_ci95,average_trace_fitness_mean,average_trace_fitness_ci95,percentage_of_fitting_traces_mean,percentage_of_fitting_traces_ci95
0,AcademicCredentials,TRAIN,0.988447,0.000000,0.988815,0.000000,84.170854,0.000000
1,BPIC_2012,TRAIN,0.999812,0.000000,0.999859,0.000000,99.669967,0.000000
2,BPIC_2017,TRAIN,0.995365,0.000000,0.993554,0.000000,89.259660,0.000000
3,AcademicCredentials,RA-RR,0.981782,0.001405,0.985680,0.001174,70.502513,2.023238
4,AcademicCredentials,DM-RR,0.979822,0.000854,0.985735,0.000820,68.869347,1.998695
5,AcademicCredentials,DM-GR,0.979453,0.000874,0.985347,0.000695,67.788945,1.462163
6,AcademicCredentials,DM-DRL,0.978974,0.001008,0.984982,0.000846,67.437186,1.647761
7,AcademicCredentials,DRL-DRL,1.000000,0.000000,1.000000,0.000000,100.000000,0.000000
8,BPIC_2012,RA-RR,1.000000,0.000000,1.000000,0.000000,100.000000,0.000000
9,BPIC_2012,DM-RR,1.000000,0.000000,1.000000,0.000000,100.000000,0.000000
